# Prophet Single-Stage Ternary Change-Point Detection


## Imports and settings


In [ ]:
!pip install prophet colorednoise scikit-learn scipy tqdm -q

import math, sys, warnings, logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

REPO_DIR = Path('.')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from generate_series import (
    iter_dataset, true_change_points, make_series,
    NOISE_CONFIGS, RANDOM_SEEDS,
)

TRAIN_PART      = 0.7
TOLERANCES      = [1, 2, 5, 10]
THRESHOLD_SIGMA = 3.0
REFRACTORY      = 20
WINDOW          = 20


NOISE_FAMILIES = ['violet', 'blue', 'white', 'pink', 'red', 'el_nino']
FAMILY_BETA = {
    'violet':  r'$\beta=-2$ (violet)',
    'blue':    r'$\beta=-1$ (blue)',
    'white':   r'$\beta=0$  (white)',
    'pink':    r'$\beta=+1$ (pink)',
    'red':     r'$\beta=+2$ (red)',
    'el_nino': r'El~Ni\~no',
}
FAMILY_BETA_PLAIN = {
    'violet':  r'$\beta=-2$' '\n(violet)',
    'blue':    r'$\beta=-1$' '\n(blue)',
    'white':   r'$\beta=0$'  '\n(white)',
    'pink':    r'$\beta=+1$' '\n(pink)',
    'red':     r'$\beta=+2$' '\n(red)',
    'el_nino': 'El Nino',
}
AMP_LABELS = {1: r'$a=1$', 2: r'$a=2$', math.pi: r'$a=\pi$'}

def noise_family(name):
    for k in NOISE_FAMILIES:
        if name.startswith(k):
            return k
    return name

def extract_amp(name):
    if name.endswith('_a1'):  return 1
    if name.endswith('_a2'):  return 2
    if name.endswith('_api'): return math.pi
    return np.nan

print(f'{len(NOISE_CONFIGS)} noise configs x {len(RANDOM_SEEDS)} seeds '
      f'= {len(NOISE_CONFIGS)*len(RANDOM_SEEDS)} series, length 10000.')


## Inline prophet ternary module (window-mean-diff statistic)


In [ ]:
from prophet import Prophet
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

DEFAULT_TOLERANCES = (1, 2, 5, 10)
DEFAULT_REFRACTORY = 20
DEFAULT_WINDOW     = 20


def prophet_fit_residuals(X, train_part=0.7,
                           changepoint_prior_scale=0.05,
                           n_changepoints=50):
    n = len(X); split = int(n * train_part)
    dates = pd.date_range('2020-01-01', periods=n, freq='D')
    df_full = pd.DataFrame({'ds': dates, 'y': X})
    m = Prophet(yearly_seasonality=False, weekly_seasonality=False,
                daily_seasonality=False,
                changepoint_prior_scale=changepoint_prior_scale,
                n_changepoints=n_changepoints)
    m.fit(df_full.iloc[:split])
    fc = m.predict(df_full[['ds']])
    resid = X - fc['yhat'].to_numpy()
    return resid, split


def robust_sigma(z):
    med = np.median(z); mad = np.median(np.abs(z - med))
    return float(max(1.4826 * mad, 1e-9))


def window_mean_diff(resid, window):
    """Vectorised forward-minus-backward window mean of `resid`.
    s_t = mean(resid[t:t+window]) - mean(resid[t-window:t]).
    Boundary windows are truncated; pre/post sample sizes used as denominators."""
    n = len(resid)
    cs = np.concatenate(([0.0], np.cumsum(resid)))
    t = np.arange(n)
    pre_lo  = np.maximum(0, t - window); pre_hi  = t
    post_lo = t;                          post_hi = np.minimum(n, t + window)
    pre_n   = np.maximum(pre_hi - pre_lo, 1)
    post_n  = np.maximum(post_hi - post_lo, 1)
    pre_mean  = (cs[pre_hi]  - cs[pre_lo])  / pre_n
    post_mean = (cs[post_hi] - cs[post_lo]) / post_n
    return post_mean - pre_mean


def nonmax_suppress_signed(raw_signed, abs_score, refractory=DEFAULT_REFRACTORY):
    n = len(raw_signed)
    y_out = np.zeros(n, dtype=int)
    cand = np.where(raw_signed != 0)[0]
    if cand.size == 0:
        return y_out
    order = cand[np.argsort(-abs_score[cand])]
    locked = np.zeros(n, dtype=bool)
    for idx in order:
        lo = max(0, idx - refractory); hi = min(n, idx + refractory + 1)
        if not locked[lo:hi].any():
            y_out[idx] = raw_signed[idx]; locked[lo:hi] = True
    return y_out


def prophet_ternary_predict(X, train_part=0.7, threshold_sigma=3.0,
                              refractory=DEFAULT_REFRACTORY,
                              window=DEFAULT_WINDOW,
                              changepoint_prior_scale=0.05,
                              n_changepoints=50):
    resid, split = prophet_fit_residuals(
        X, train_part=train_part,
        changepoint_prior_scale=changepoint_prior_scale,
        n_changepoints=n_changepoints)
    score = window_mean_diff(resid, window=window)
    sigma_s = robust_sigma(score[:split])
    tau = threshold_sigma * sigma_s
    raw = np.zeros(len(X), dtype=int)
    raw[score >  tau] = +1
    raw[score < -tau] = -1
    y_hat = nonmax_suppress_signed(raw, np.abs(score), refractory)
    return {'y_hat': y_hat, 'resid': resid, 'score': score,
            'sigma': sigma_s, 'tau': tau, 'split': split}


def state_to_cps(y):
    y = np.asarray(y, dtype=int)
    if y.size == 0:
        return np.array([], dtype=int), np.array([], dtype=int)
    diff = np.diff(y); pos = np.where(diff != 0)[0] + 1
    if pos.size == 0:
        return pos, np.array([], dtype=int)
    return pos, np.sign(diff[pos - 1]).astype(int)


def events_to_cps(y_events):
    y_events = np.asarray(y_events, dtype=int)
    pos = np.where(y_events != 0)[0]
    return pos, y_events[pos].astype(int)


def signed_match(true_cps, true_signs, pred_cps, pred_signs, tolerance):
    true_cps = np.asarray(true_cps, dtype=int); pred_cps = np.asarray(pred_cps, dtype=int)
    true_signs = np.asarray(true_signs, dtype=int); pred_signs = np.asarray(pred_signs, dtype=int)
    if true_cps.size == 0 and pred_cps.size == 0: return 0, 0, 0, 0
    if true_cps.size == 0:  return 0, 0, int(pred_cps.size), 0
    if pred_cps.size == 0:  return 0, 0, 0, int(true_cps.size)
    o_t = np.argsort(true_cps); o_p = np.argsort(pred_cps)
    true_cps = true_cps[o_t]; true_signs = true_signs[o_t]
    pred_cps = pred_cps[o_p]; pred_signs = pred_signs[o_p]
    matched = np.zeros(true_cps.size, dtype=bool)
    tp_d = 0; tp_s = 0
    for pi, p in enumerate(pred_cps):
        d = np.abs(true_cps - p)
        cand = np.where((d <= tolerance) & (~matched))[0]
        if cand.size:
            best = cand[np.argmin(d[cand])]
            matched[best] = True; tp_d += 1
            if pred_signs[pi] == true_signs[best]:
                tp_s += 1
    return int(tp_d), int(tp_s), int(pred_cps.size - tp_d), int(true_cps.size - matched.sum())


def _pr_f1(tp, fp, fn):
    p = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    r = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    return f, p, r


def events_to_state_path(y_events, min_state=-1, max_state=+1, initial_state=0):
    y_events = np.asarray(y_events, dtype=int)
    out = np.zeros_like(y_events); s = int(initial_state)
    for t in range(len(y_events)):
        if y_events[t] != 0:
            s = max(min_state, min(max_state, s + int(y_events[t])))
        out[t] = s
    return out


def ternary_metrics(y_true, y_pred, tolerances=DEFAULT_TOLERANCES, pred_format='events'):
    y_true = np.asarray(y_true, dtype=int); y_pred = np.asarray(y_pred, dtype=int)
    true_cps, true_signs = state_to_cps(y_true)
    if pred_format == 'events':
        pred_cps, pred_signs = events_to_cps(y_pred)
    else:
        pred_cps, pred_signs = state_to_cps(y_pred)
    out = {}
    for k in tolerances:
        tp_d, tp_s, fp, fn = signed_match(true_cps, true_signs, pred_cps, pred_signs, k)
        f1_d, p_d, r_d = _pr_f1(tp_d, fp, fn)
        f1_s, p_s, r_s = _pr_f1(tp_s, fp, fn)
        out[f'tol_{k}_f1_det']           = f1_d
        out[f'tol_{k}_precision_det']    = p_d
        out[f'tol_{k}_recall_det']       = r_d
        out[f'tol_{k}_f1_signed']        = f1_s
        out[f'tol_{k}_precision_signed'] = p_s
        out[f'tol_{k}_recall_signed']    = r_s
        out[f'tol_{k}_dir_acc']          = (tp_s/tp_d) if tp_d > 0 else np.nan
        out[f'tol_{k}_tp_det']           = tp_d
        out[f'tol_{k}_tp_signed']        = tp_s
        out[f'tol_{k}_fp']               = fp
        out[f'tol_{k}_fn']               = fn
    y_pred_state = events_to_state_path(y_pred) if pred_format == 'events' else y_pred
    out['balanced_acc'] = float(balanced_accuracy_score(y_true, y_pred_state))
    cm = confusion_matrix(y_true, y_pred_state, labels=[-1, 0, 1])
    for i, ci in enumerate([-1, 0, 1]):
        for j, cj in enumerate([-1, 0, 1]):
            out[f'cm_{ci}_{cj}'] = int(cm[i, j])
    row_tot = cm.sum(axis=1)
    for i, ci in enumerate([-1, 0, 1]):
        out[f'recall_class_{ci}'] = float(cm[i, i] / row_tot[i]) if row_tot[i] > 0 else np.nan
    return out


def run_prophet_on_series(X, y, train_part=0.7, threshold_sigma=3.0,
                            refractory=DEFAULT_REFRACTORY, window=DEFAULT_WINDOW,
                            tolerances=DEFAULT_TOLERANCES):
    pred = prophet_ternary_predict(X, train_part=train_part,
                                     threshold_sigma=threshold_sigma,
                                     refractory=refractory, window=window)
    m = ternary_metrics(y, pred['y_hat'], tolerances=tolerances)
    m['sigma'] = pred['sigma']; m['tau'] = pred['tau']
    m['n_pred_cps'] = int((pred['y_hat'] != 0).sum())
    return m


print('Detector and metric helpers defined.')


## Smoke test on one series


In [ ]:
noise_name, seed = 'red_a2', 0
df_demo = make_series(noise_name, seed)
X = df_demo['x'].to_numpy()
y = df_demo['state'].to_numpy()

pred = prophet_ternary_predict(X, train_part=TRAIN_PART,
                                 threshold_sigma=THRESHOLD_SIGMA,
                                 refractory=REFRACTORY, window=WINDOW)
y_hat = pred['y_hat']; score = pred['score']
sigma = pred['sigma']; tau = pred['tau']; split = pred['split']

metrics = ternary_metrics(y, y_hat, tolerances=TOLERANCES)
print(f'F1_signed (k=5) = {metrics["tol_5_f1_signed"]:.3f}, BalAcc = {metrics["balanced_acc"]:.3f}')

true_cps = true_change_points(y)
pred_pos = np.where(y_hat > 0)[0]
pred_neg = np.where(y_hat < 0)[0]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax = axes[0]
ax.plot(X, color='steelblue', lw=0.5, label='signal X')
ax.plot(df_demo['level'].to_numpy(), color='orange', lw=1.2, label='true level')
for cp in true_cps:
    ax.axvline(cp, color='red', ls='--', lw=0.5, alpha=0.5)
ax.set_title(f'Signal and true levels - {noise_name}, seed={seed}')
ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.2)

ax = axes[1]
ax.plot(score, color='gray', lw=0.5, label='$s_t = \\bar e^+_t - \\bar e^-_t$')
ax.scatter(pred_pos, score[pred_pos], color='green', s=24, zorder=5, label='predicted +1')
ax.scatter(pred_neg, score[pred_neg], color='red',   s=24, zorder=5, label='predicted -1')
for cp in true_cps:
    ax.axvline(cp, color='red', ls='--', lw=0.4, alpha=0.4)
ax.set_title('Window-mean-diff statistic')
ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('prophet_smoke_test.png', dpi=140, bbox_inches='tight')
plt.show()


## Training and calibration


In [ ]:
yhat_trend = X - pred['resid']
score_train = score[:split]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(X, color='steelblue', lw=0.5, label='signal')
axes[0].plot(yhat_trend, color='goldenrod', lw=1.2, label='Prophet trend')
axes[0].axvspan(0, split, color='steelblue', alpha=0.08, label='train')
axes[0].set_xlabel('t'); axes[0].set_ylabel('X')
axes[0].set_title('Prophet trend')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(score_train, bins=60, color='steelblue', alpha=0.7)
axes[1].axvline( tau, color='indianred', ls='--', label=f'tau = {tau:.2f}')
axes[1].axvline(-tau, color='indianred', ls='--')
axes[1].set_xlabel('score s_t (train)'); axes[1].set_ylabel('count')
axes[1].set_title('Score distribution and threshold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('prophet_training_calibration.png', dpi=140, bbox_inches='tight')
plt.show()


## Full benchmark loop


In [ ]:
rows = []
_first_exc = None

for noise_name, seed, df in tqdm(iter_dataset(),
                                   total=len(NOISE_CONFIGS)*len(RANDOM_SEEDS),
                                   desc='Prophet ternary'):
    X = df['x'].to_numpy(); y = df['state'].to_numpy()
    try:
        pred = prophet_ternary_predict(X, train_part=TRAIN_PART,
                                          threshold_sigma=THRESHOLD_SIGMA,
                                          refractory=REFRACTORY, window=WINDOW)
        split = pred['split']; y_hat = pred['y_hat']
        m_full = ternary_metrics(y, y_hat, tolerances=TOLERANCES)
        m_train = ternary_metrics(y[:split], y_hat[:split], tolerances=TOLERANCES)
        m_test  = ternary_metrics(y[split:], y_hat[split:], tolerances=TOLERANCES)
        m = {**m_full,
             **{f'train_{k}': v for k, v in m_train.items()},
             **{f'test_{k}': v  for k, v in m_test.items()}}
        m['sigma']     = pred['sigma']; m['tau'] = pred['tau']
        m['n_pred_cps'] = int((y_hat != 0).sum())
        m['n_true_cps'] = int(((np.diff(y) != 0).sum()))
        m['n_pred_train'] = int((y_hat[:split] != 0).sum())
        m['n_pred_test']  = int((y_hat[split:] != 0).sum())
    except Exception as e:
        if _first_exc is None: _first_exc = e
        m = {}
    rows.append({'noise': noise_name, 'seed': seed,
                  'noise_family': noise_family(noise_name),
                  'amplitude': extract_amp(noise_name), **m})
if _first_exc:
    import traceback
    traceback.print_exception(type(_first_exc), _first_exc, _first_exc.__traceback__)

df_prophet = pd.DataFrame(rows)
df_prophet.to_csv('prophet_ternary_results.csv', index=False)
print(f'Done: {len(df_prophet)} rows, {df_prophet["tol_5_f1_det"].notna().sum()} valid.')
df_prophet.head(3)


## In-sample vs out-of-sample sanity check


In [ ]:
tol = 5
cols = ['n_true_cps', 'n_pred_cps', 'n_pred_train', 'n_pred_test',
        f'tol_{tol}_f1_det',  f'train_tol_{tol}_f1_det',  f'test_tol_{tol}_f1_det',
        f'tol_{tol}_f1_signed', f'train_tol_{tol}_f1_signed', f'test_tol_{tol}_f1_signed',
        'balanced_acc', 'train_balanced_acc', 'test_balanced_acc']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
metrics = [
    (f'tol_{tol}_f1_det',    f'$F_1^{{\\mathrm{{det}}}}$ at $k = {tol}$'),
    (f'tol_{tol}_f1_signed', f'$F_1^{{\\mathrm{{sgn}}}}$ at $k = {tol}$'),
    ('balanced_acc',         'Balanced accuracy'),
]
x = np.arange(len(NOISE_FAMILIES))
w = 0.27
for ax, (col, label) in zip(axes, metrics):
    fam_means = {scope: [] for scope in ['full', 'train', 'test']}
    fam_ses   = {scope: [] for scope in ['full', 'train', 'test']}
    for fam in NOISE_FAMILIES:
        sub = df_prophet[df_prophet['noise_family'] == fam]
        for scope, prefix in [('full', ''), ('train', 'train_'), ('test', 'test_')]:
            colname = prefix + col
            v = sub[colname].dropna()
            fam_means[scope].append(v.mean() if len(v) else np.nan)
            fam_ses[scope].append((v.std(ddof=1)/np.sqrt(len(v))) if len(v) > 1 else 0)
    ax.bar(x - w, fam_means['full'],  w, yerr=fam_ses['full'],  color='gray',     label='Full series', capsize=3)
    ax.bar(x,     fam_means['train'], w, yerr=fam_ses['train'], color='steelblue', label='Train range (in-sample)',  capsize=3)
    ax.bar(x + w, fam_means['test'],  w, yerr=fam_ses['test'],  color='indianred', label='Test range (out-of-sample)', capsize=3)
    ax.set_xticks(x); ax.set_xticklabels([FAMILY_BETA_PLAIN[f] for f in NOISE_FAMILIES], fontsize=8)
    ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=0.3)
    ax.set_title(label, fontsize=11)
axes[0].legend(loc='lower left', fontsize=9, frameon=False)
fig.suptitle('In-sample (train range) vs out-of-sample (test range) performance.\n'
             'A small gap means the detector is not exploiting Prophet\'s in-sample changepoint absorption.',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('prophet_train_vs_test_gap.png', dpi=140, bbox_inches='tight')
plt.show()

delta_det    = (df_prophet[f'train_tol_{tol}_f1_det']    - df_prophet[f'test_tol_{tol}_f1_det']).dropna()
delta_signed = (df_prophet[f'train_tol_{tol}_f1_signed'] - df_prophet[f'test_tol_{tol}_f1_signed']).dropna()
delta_ba     = (df_prophet['train_balanced_acc']         - df_prophet['test_balanced_acc']).dropna()
print(f'train - test gap: F1_det={delta_det.mean():.3f}, F1_signed={delta_signed.mean():.3f}, BalAcc={delta_ba.mean():.3f}')


## Random-baseline F1


In [ ]:
tol = 5
T = 10000
rng = np.random.default_rng(0)

baseline_rows = []
for _, r in df_prophet.iterrows():
    if not np.isfinite(r['n_pred_cps']) or not np.isfinite(r['n_true_cps']):
        continue
    n_pred = int(r['n_pred_cps']); n_true = int(r['n_true_cps'])
    if n_pred == 0 or n_true == 0:
        continue
    pred_pos = rng.choice(T, size=n_pred, replace=False)
    pred_sgn = rng.choice([-1, +1], size=n_pred)
    true_pos = rng.choice(T, size=n_true, replace=False)
    true_sgn = rng.choice([-1, +1], size=n_true)
    tp_d, tp_s, fp, fn = signed_match(true_pos, true_sgn, pred_pos, pred_sgn, tol)
    f1_d, _, _ = _pr_f1(tp_d, fp, fn)
    f1_s, _, _ = _pr_f1(tp_s, fp, fn)
    baseline_rows.append({'noise_family': r['noise_family'],
                            'baseline_f1_det': f1_d, 'baseline_f1_signed': f1_s,
                            'prophet_f1_det': r[f'tol_{tol}_f1_det'],
                            'prophet_f1_signed': r[f'tol_{tol}_f1_signed']})
df_baseline = pd.DataFrame(baseline_rows)

summary = df_baseline.groupby('noise_family')[
    ['prophet_f1_det', 'baseline_f1_det', 'prophet_f1_signed', 'baseline_f1_signed']
].mean().reindex(NOISE_FAMILIES).round(3)
summary['lift_det']    = (summary['prophet_f1_det']    - summary['baseline_f1_det']).round(3)
summary['lift_signed'] = (summary['prophet_f1_signed'] - summary['baseline_f1_signed']).round(3)
print(summary)

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(NOISE_FAMILIES)); w = 0.35
ax.bar(x - w/2, summary['prophet_f1_signed'],  w, color='steelblue', label='Prophet (signed $F_1$)')
ax.bar(x + w/2, summary['baseline_f1_signed'], w, color='lightgray', label='Random baseline')
ax.set_xticks(x); ax.set_xticklabels([FAMILY_BETA_PLAIN[f] for f in NOISE_FAMILIES], fontsize=9)
ax.set_ylim(0, 1.05); ax.set_ylabel(f'Signed $F_1$ at $k = {tol}$', fontsize=11)
ax.set_title('Prophet detector vs random baseline matched on number of firings\n'
             '(random detector fires the same n_pred per series at uniform positions and random signs)',
             fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('prophet_vs_random_baseline.png', dpi=140, bbox_inches='tight')
plt.show()


## Sensitivity to the window horizon h


In [ ]:
H_GRID = [5, 10, 20, 30]
sens_rows = []
first_exc_sens = None

for noise_name, seed, df in tqdm(iter_dataset(),
                                   total=len(NOISE_CONFIGS)*len(RANDOM_SEEDS),
                                   desc='Sensitivity (h sweep, cached Prophet fit)'):
    X = df['x'].to_numpy(); y = df['state'].to_numpy()
    try:
        resid, split = prophet_fit_residuals(X, train_part=TRAIN_PART)
    except Exception as e:
        if first_exc_sens is None: first_exc_sens = e
        for h in H_GRID:
            sens_rows.append({'noise': noise_name, 'seed': seed, 'h': h,
                                'noise_family': noise_family(noise_name)})
        continue
    for h in H_GRID:
        score = window_mean_diff(resid, window=h)
        sigma_s = robust_sigma(score[:split])
        tau = THRESHOLD_SIGMA * sigma_s
        raw = np.zeros(len(X), dtype=int)
        raw[score >  tau] = +1
        raw[score < -tau] = -1
        y_hat = nonmax_suppress_signed(raw, np.abs(score), REFRACTORY)
        m = ternary_metrics(y, y_hat, tolerances=[5])
        sens_rows.append({'noise': noise_name, 'seed': seed, 'h': h,
                            'noise_family': noise_family(noise_name),
                            'f1_det':    m['tol_5_f1_det'],
                            'f1_signed': m['tol_5_f1_signed'],
                            'balanced_acc': m['balanced_acc'],
                            'dir_acc':   m['tol_5_dir_acc']})

df_sens = pd.DataFrame(sens_rows)
df_sens.to_csv('prophet_h_sensitivity.csv', index=False)
if first_exc_sens is not None:
    print(f'[warn] first fit exception: {first_exc_sens!r}')

family_colors = {
    'violet':  '#8e44ad',
    'blue':    '#3498db',
    'white':   '#7f8c8d',
    'pink':    '#e91e63',
    'red':     '#c0392b',
    'el_nino': '#16a085',
}

sens_summary = (df_sens.groupby(['noise_family', 'h'])[['f1_signed', 'balanced_acc']]
                 .agg(['mean', 'std', 'count']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, metric, ylabel in zip(
    axes,
    ['f1_signed', 'balanced_acc'],
    [f'$F_{{1,\\mathrm{{signed}}}}$  (tolerance $k = 5$)',
     'Balanced accuracy (3-class macro recall)']):
    for fam in NOISE_FAMILIES:
        mus, ses, hs_have = [], [], []
        for h in H_GRID:
            s = df_sens[(df_sens['noise_family'] == fam) & (df_sens['h'] == h)][metric].dropna()
            if len(s) == 0: continue
            mus.append(s.mean()); ses.append(s.std(ddof=1) / np.sqrt(len(s))); hs_have.append(h)
        ax.errorbar(hs_have, mus, yerr=ses, marker='o', lw=1.8, capsize=4,
                    color=family_colors[fam], label=FAMILY_BETA_PLAIN[fam].replace('\n', ' '))
    ax.set_xlabel('Window horizon $h$', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xticks(H_GRID)
    ax.set_ylim(0, 1.0)
    ax.grid(alpha=0.3)
    if metric == 'balanced_acc':
        ax.axhline(1/3, color='gray', ls=':', lw=1, alpha=0.7)
axes[0].legend(loc='lower right', fontsize=9, ncol=2, frameon=False)
axes[1].legend(loc='lower right', fontsize=9, ncol=2, frameon=False)
fig.suptitle('Prophet single-stage detector: sensitivity to window horizon $h$.\n'
             'Each curve is one noise family; $h = 20$ is the canonical setting.',
             y=1.02, fontsize=11)
plt.tight_layout()
plt.savefig('prophet_h_sensitivity.png', dpi=140, bbox_inches='tight')
plt.show()

tbl_f1 = (df_sens.groupby(['noise_family', 'h'])['f1_signed'].mean()
          .unstack('h').reindex(NOISE_FAMILIES).round(3))
tbl_ba = (df_sens.groupby(['noise_family', 'h'])['balanced_acc'].mean()
          .unstack('h').reindex(NOISE_FAMILIES).round(3))
print(tbl_f1)
print(tbl_ba)


## Signed $F_1$ and balanced accuracy by noise family


In [ ]:
def family_mean_se(df, col):
    out = []
    for fam in NOISE_FAMILIES:
        s = df[df['noise_family'] == fam][col].dropna()
        if len(s) == 0:
            out.append((np.nan, np.nan)); continue
        out.append((s.mean(), s.std(ddof=1) / np.sqrt(len(s))))
    return np.array(out)

tol = 5
sgn = family_mean_se(df_prophet, f'tol_{tol}_f1_signed')
ba  = family_mean_se(df_prophet, 'balanced_acc')
labels = [FAMILY_BETA_PLAIN[f] for f in NOISE_FAMILIES]
x = np.arange(len(NOISE_FAMILIES))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.bar(x, sgn[:, 0], yerr=sgn[:, 1], color='steelblue', edgecolor='black',
       capsize=4, width=0.7)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel(f'$F_{{1,\\mathrm{{signed}}}}$  (tolerance $k = {tol}$)', fontsize=11)
ax.set_title(f'Signed $F_1$  (tolerance $k = {tol}$)', fontsize=12)
ax.grid(axis='y', alpha=0.3)
ax = axes[1]
ax.bar(x, ba[:, 0], yerr=ba[:, 1], color='indianred', edgecolor='black',
       capsize=4, width=0.7)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Balanced accuracy (3-class macro recall)', fontsize=11)
ax.set_title('Pointwise balanced accuracy', fontsize=12)
ax.axhline(1/3, color='gray', ls='--', lw=1, label='1/3 floor')
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='upper right', fontsize=9)
fig.suptitle(f'Prophet single-stage detector: signed $F_1$ (left, $k = {tol}$) and balanced accuracy (right)\n'
              f'by noise family. Mean$\\pm$SE across 30 series per family.',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('prophet_fig4_family_bars.png', dpi=140, bbox_inches='tight')
plt.show()


## Detection $F_1$, signed $F_1$, conditional direction accuracy and balanced accuracy vs tolerance


In [ ]:
def overall_mean_se(df, col):
    s = df[col].dropna()
    if len(s) == 0: return (np.nan, np.nan)
    return (s.mean(), s.std(ddof=1) / np.sqrt(len(s)))

tols = TOLERANCES
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
panels = [
    ('tol_{k}_f1_det',    '$F_1^{\\mathrm{det}}$ (position-only)',          'steelblue'),
    ('tol_{k}_f1_signed', '$F_1^{\\mathrm{sgn}}$ (position $\\wedge$ sign)', 'indianred'),
    ('tol_{k}_dir_acc',   'Conditional direction accuracy',                    'goldenrod'),
    ('balanced_acc',      'Balanced accuracy (3-class macro recall)',          'forestgreen'),
]
for ax, (tmpl, ylab, c) in zip(axes.ravel(), panels):
    if 'balanced' in tmpl:
        mu, se = overall_mean_se(df_prophet, 'balanced_acc')
        ax.errorbar(tols, [mu]*len(tols), yerr=[se]*len(tols),
                    color=c, marker='o', lw=1.6, capsize=4)
        ax.axhline(1/3, color='gray', ls='--', lw=1)
    else:
        mus, ses = [], []
        for k in tols:
            mu, se = overall_mean_se(df_prophet, tmpl.format(k=k))
            mus.append(mu); ses.append(se)
        ax.errorbar(tols, mus, yerr=ses, color=c, marker='o', lw=1.6, capsize=4)
        if 'dir_acc' in tmpl:
            ax.axhline(0.5, color='gray', ls='--', lw=1, label='chance')
            ax.legend(loc='lower right', fontsize=9)
    ax.set_xticks(tols)
    ax.set_xlabel('Tolerance $k$', fontsize=11)
    ax.set_ylabel(ylab, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)

fig.suptitle('Prophet single-stage detector: four-panel metric grid vs tolerance $k$.\n'
             'Mean$\\pm$SE across all 180 series.', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('prophet_fig5_metric_panel.png', dpi=140, bbox_inches='tight')
plt.show()


## Row-normalised 3x3 confusion matrices per noise family


In [ ]:
labels_axis = [-1, 0, 1]
tick_x = [r'$\widehat y=-1$', r'$\widehat y=0$', r'$\widehat y=+1$']
tick_y = [r'$y=-1$',           r'$y=0$',           r'$y=+1$']

fig, axes = plt.subplots(1, 6, figsize=(20, 3.6))
for ax, fam in zip(axes, NOISE_FAMILIES):
    sub = df_prophet[df_prophet['noise_family'] == fam]
    cm_sum = np.array([[sub[f'cm_{ci}_{cj}'].sum() for cj in labels_axis]
                       for ci in labels_axis], dtype=float)
    row_tot = cm_sum.sum(axis=1, keepdims=True)
    norm = np.where(row_tot > 0, cm_sum / row_tot, 0.0)
    im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(tick_x, fontsize=8)
    ax.set_yticks(range(3)); ax.set_yticklabels(tick_y, fontsize=8)
    ax.set_title(FAMILY_BETA[fam], fontsize=11)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{norm[i,j]:.2f}', ha='center', va='center',
                    color='black' if norm[i,j] < 0.55 else 'white', fontsize=10)
fig.subplots_adjust(right=0.92, wspace=0.3)
cbar_ax = fig.add_axes([0.94, 0.18, 0.012, 0.65])
fig.colorbar(im, cax=cbar_ax, label='Row-normalised proportion')
fig.suptitle('Prophet single-stage detector: row-normalised 3x3 confusion matrices per noise family.\n'
             'Aggregated over amplitude and seed (30 series per family).',
             y=1.05, fontsize=12)
plt.savefig('prophet_fig6_confusion_row.png', dpi=140, bbox_inches='tight')
plt.show()


## Signed $F_1$ and balanced accuracy vs amplitude, faceted by noise family


In [ ]:
amps = sorted(df_prophet['amplitude'].dropna().unique())
amp_x = list(range(len(amps)))
amp_labels = [AMP_LABELS.get(a, str(a)) for a in amps]
tol = 5

def amp_curve(sub):
    sgn = []; ba = []
    for a in amps:
        s = sub[sub['amplitude'] == a]
        sgn.append(s[f'tol_{tol}_f1_signed'].mean() if len(s) else np.nan)
        ba.append( s['balanced_acc'].mean()         if len(s) else np.nan)
    return sgn, ba

fig, axes = plt.subplots(1, 7, figsize=(20, 3.6), sharey=True)
panels = list(NOISE_FAMILIES) + ['__overall__']
for ax, fam in zip(axes, panels):
    if fam == '__overall__':
        sgn, ba = amp_curve(df_prophet); title = 'Overall'
    else:
        sgn, ba = amp_curve(df_prophet[df_prophet['noise_family'] == fam])
        title = FAMILY_BETA[fam]
    ax.plot(amp_x, sgn, marker='o', color='steelblue', lw=1.6,
            label=f'$F_{{1,\\mathrm{{signed}}}}$ ($k={tol}$)')
    ax.plot(amp_x, ba,  marker='s', color='indianred', lw=1.6,
            label='Balanced acc.')
    ax.set_xticks(amp_x); ax.set_xticklabels(amp_labels, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.set_xlabel('Amplitude $a$', fontsize=10)
    ax.grid(alpha=0.3)
    ax.axhline(1/3, color='gray', ls='--', lw=0.8)
axes[0].set_ylabel(f'$F_{{1,\\mathrm{{signed}}}}$ and balanced accuracy', fontsize=10)
axes[-1].legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
fig.suptitle('Prophet single-stage detector: signed $F_1$ (blue) and balanced accuracy (red) as a function of amplitude $a$,\n'
             'faceted by noise family; "Overall" pools all six families.',
             y=1.05, fontsize=12)
plt.tight_layout()
plt.savefig('prophet_fig7_amp_facet.png', dpi=140, bbox_inches='tight')
plt.show()


## Example series with detected and true change points


In [ ]:
examples = [
    ('red_a1',    7),
    ('red_api',   0),
    ('red_a2',    3),
    ('blue_a2',   4),
    ('white_a1',  1),
    ('el_nino_a1', 0),
]

fig, axes = plt.subplots(len(examples), 1, figsize=(13, 2.0 * len(examples)),
                            sharex=True)
for ax, (name, seed) in zip(axes, examples):
    df = make_series(name, seed)
    X = df['x'].to_numpy(); y = df['state'].to_numpy()
    pred = prophet_ternary_predict(X, train_part=TRAIN_PART,
                                     threshold_sigma=THRESHOLD_SIGMA,
                                     refractory=REFRACTORY, window=WINDOW)
    metrics = ternary_metrics(y, pred['y_hat'], tolerances=[5])
    tcp = true_change_points(y)
    pp = np.where(pred['y_hat'] > 0)[0]; pn = np.where(pred['y_hat'] < 0)[0]
    da_str = (f'{metrics["tol_5_dir_acc"]:.2f}'
              if metrics['tol_5_dir_acc'] == metrics['tol_5_dir_acc'] else 'nan')
    ax.plot(X, color='steelblue', lw=0.5)
    for cp in tcp:
        ax.axvline(cp, color='red', ls='--', lw=0.4, alpha=0.5)
    if len(pp): ax.scatter(pp, X[pp], color='green', s=18, zorder=5, marker='^')
    if len(pn): ax.scatter(pn, X[pn], color='red',   s=18, zorder=5, marker='v')
    fam = noise_family(name); amp = extract_amp(name)
    ax.set_title(f'{FAMILY_BETA[fam]}, $a={AMP_LABELS.get(amp,amp)[3:-1]}$, seed={seed} | '
                  f'$F_1^{{det}}={metrics["tol_5_f1_det"]:.2f}$, '
                  f'$F_1^{{sgn}}={metrics["tol_5_f1_signed"]:.2f}$, '
                  f'DirAcc$={da_str}$', fontsize=9)
    ax.grid(alpha=0.2)
axes[-1].set_xlabel('$t$', fontsize=11)
plt.tight_layout()
plt.savefig('prophet_fig8_examples.png', dpi=140, bbox_inches='tight')
plt.show()


## How the detection statistic encodes both position and direction


In [ ]:
showcase_name, showcase_seed = 'red_a1', 7
df_s = make_series(showcase_name, showcase_seed)
X = df_s['x'].to_numpy(); y = df_s['state'].to_numpy()
level = df_s['level'].to_numpy()

pred = prophet_ternary_predict(X, train_part=TRAIN_PART,
                                 threshold_sigma=THRESHOLD_SIGMA,
                                 refractory=REFRACTORY, window=WINDOW)
resid = pred['resid']; score = pred['score']
sigma_s = pred['sigma']; tau = pred['tau']; split = pred['split']; y_hat = pred['y_hat']

true_cps = true_change_points(y)
true_signs = np.sign(np.diff(y))[true_cps - 1].astype(int)
pos_cps = true_cps[true_signs > 0]
neg_cps = true_cps[true_signs < 0]
pred_pos = np.where(y_hat > 0)[0]
pred_neg = np.where(y_hat < 0)[0]

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True,
                            gridspec_kw={'height_ratios': [1.2, 1, 1.2]})

# Row 1: signal X_t + Prophet trend
ax = axes[0]
yhat_trend = X - resid  # X - resid = yhat_prophet
ax.plot(X, color='steelblue', lw=0.5, alpha=0.9, label='$X_t$ (signal)')
ax.plot(yhat_trend, color='goldenrod', lw=1.0, alpha=0.9, label='$\\widehat{y}^{\\mathrm{Prophet}}_t$ (trend)')
ax.plot(level, color='dimgray', lw=1.2, alpha=0.6, label='true level $L_t$')
for cp in pos_cps: ax.axvline(cp, color='green', ls='--', lw=0.5, alpha=0.6)
for cp in neg_cps: ax.axvline(cp, color='crimson', ls='--', lw=0.5, alpha=0.6)
ax.set_ylabel('signal', fontsize=11)
ax.set_title(f'(a) signal $X_t$ with Prophet trend and true level - {showcase_name}, seed={showcase_seed}',
             fontsize=11)
ax.legend(loc='upper right', fontsize=9, ncol=3); ax.grid(alpha=0.2)

# Row 2: residual
ax = axes[1]
ax.plot(resid, color='gray', lw=0.4)
for cp in pos_cps: ax.axvline(cp, color='green', ls='--', lw=0.4, alpha=0.6)
for cp in neg_cps: ax.axvline(cp, color='crimson', ls='--', lw=0.4, alpha=0.6)
ax.set_ylabel('$e_t$ (residual)', fontsize=11)
ax.set_title('(b) Prophet residual $e_t = X_t - \\widehat{y}^{\\mathrm{Prophet}}_t$', fontsize=11)
ax.grid(alpha=0.2)

# Row 3: score s_t with band and predictions
ax = axes[2]
ax.plot(score, color='steelblue', lw=0.5)
ax.scatter(pred_pos, score[pred_pos], color='green',   s=28, zorder=5,
           marker='^', label=f'predicted $\\widehat y_t = +1$ ({len(pred_pos)})')
ax.scatter(pred_neg, score[pred_neg], color='crimson', s=28, zorder=5,
           marker='v', label=f'predicted $\\widehat y_t = -1$ ({len(pred_neg)})')
for cp in pos_cps: ax.axvline(cp, color='green',   ls='--', lw=0.4, alpha=0.5)
for cp in neg_cps: ax.axvline(cp, color='crimson', ls='--', lw=0.4, alpha=0.5)
ax.set_ylabel('$s_t$ (score)', fontsize=11)
ax.set_xlabel('$t$', fontsize=11)
ax.set_title('(c) window-mean-diff statistic $s_t = \\bar e^+_t - \\bar e^-_t$  '
             '($h = ' f'{WINDOW}' '$); sharp signed peaks at every level jump',
             fontsize=11)
ax.legend(loc='upper right', fontsize=9, ncol=2); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('prophet_fig9_score_overlay.png', dpi=140, bbox_inches='tight')
plt.show()

m = ternary_metrics(y, y_hat, tolerances=[5])
print(f'F1_signed = {m["tol_5_f1_signed"]:.3f}, BalAcc = {m["balanced_acc"]:.3f}')


## Prophet vs single-stage GMM vs two-stage GMM cascade


In [ ]:
PAPER_TABLE2 = {
    'violet':  {'gmm_single_f1': 0.064, 'gmm_single_ba': 0.363,
                'gmm_cascade_f1': 0.263, 'gmm_cascade_ba': 0.453},
    'blue':    {'gmm_single_f1': 0.064, 'gmm_single_ba': 0.362,
                'gmm_cascade_f1': 0.202, 'gmm_cascade_ba': 0.420},
    'white':   {'gmm_single_f1': 0.064, 'gmm_single_ba': 0.363,
                'gmm_cascade_f1': 0.090, 'gmm_cascade_ba': 0.367},
    'pink':    {'gmm_single_f1': 0.029, 'gmm_single_ba': 0.374,
                'gmm_cascade_f1': 0.119, 'gmm_cascade_ba': 0.378},
    'red':     {'gmm_single_f1': 0.017, 'gmm_single_ba': 0.380,
                'gmm_cascade_f1': 0.706, 'gmm_cascade_ba': 0.842},
    'el_nino': {'gmm_single_f1': 0.045, 'gmm_single_ba': 0.370,
                'gmm_cascade_f1': 0.015, 'gmm_cascade_ba': 0.338},
}

tol = 5
prophet_f1 = []; prophet_ba = []
gmm_s_f1   = []; gmm_s_ba   = []
gmm_c_f1   = []; gmm_c_ba   = []
for fam in NOISE_FAMILIES:
    sub = df_prophet[df_prophet['noise_family'] == fam]
    prophet_f1.append(sub[f'tol_{tol}_f1_signed'].mean())
    prophet_ba.append(sub['balanced_acc'].mean())
    gmm_s_f1.append(PAPER_TABLE2[fam]['gmm_single_f1'])
    gmm_s_ba.append(PAPER_TABLE2[fam]['gmm_single_ba'])
    gmm_c_f1.append(PAPER_TABLE2[fam]['gmm_cascade_f1'])
    gmm_c_ba.append(PAPER_TABLE2[fam]['gmm_cascade_ba'])

x = np.arange(len(NOISE_FAMILIES)); w = 0.27
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

ax = axes[0]
ax.bar(x - w, gmm_s_f1,   w, color='lightgray', edgecolor='black', label='Single-stage GMM (paper)')
ax.bar(x,     gmm_c_f1,   w, color='indianred', edgecolor='black', label='Two-stage GMM cascade (paper)')
ax.bar(x + w, prophet_f1, w, color='steelblue', edgecolor='black', label='Prophet single-stage (this)')
ax.set_xticks(x); ax.set_xticklabels([FAMILY_BETA_PLAIN[f] for f in NOISE_FAMILIES], fontsize=9)
ax.set_ylabel(f'$F_{{1,\\mathrm{{signed}}}}$  (tolerance $k={tol}$)', fontsize=11)
ax.set_ylim(0, 1.0); ax.grid(axis='y', alpha=0.3)
ax.set_title(f'Signed $F_1$ at $k = {tol}$', fontsize=12)
ax.legend(loc='upper left', fontsize=9, frameon=False)

ax = axes[1]
ax.bar(x - w, gmm_s_ba,   w, color='lightgray', edgecolor='black', label='Single-stage GMM (paper)')
ax.bar(x,     gmm_c_ba,   w, color='indianred', edgecolor='black', label='Two-stage GMM cascade (paper)')
ax.bar(x + w, prophet_ba, w, color='steelblue', edgecolor='black', label='Prophet single-stage (this)')
ax.set_xticks(x); ax.set_xticklabels([FAMILY_BETA_PLAIN[f] for f in NOISE_FAMILIES], fontsize=9)
ax.set_ylabel('Balanced accuracy (3-class macro recall)', fontsize=11)
ax.set_ylim(0, 1.0); ax.grid(axis='y', alpha=0.3)
ax.axhline(1/3, color='gray', ls='--', lw=1, label='1/3 floor (y=0 everywhere)')
ax.set_title('Balanced accuracy', fontsize=12)
ax.legend(loc='upper left', fontsize=9, frameon=False)

fig.suptitle('Head-to-head: Prophet single-stage detector vs. single-stage GMM baseline and two-stage GMM cascade.\n',
             y=1.04, fontsize=12)
plt.tight_layout()
plt.savefig('prophet_fig10_gmm_comparison.png', dpi=140, bbox_inches='tight')
plt.show()

comp = pd.DataFrame({
    'noise_family':      NOISE_FAMILIES,
    'GMM_single_F1':     np.round(gmm_s_f1, 3),
    'GMM_cascade_F1':    np.round(gmm_c_f1, 3),
    'Prophet_F1':        np.round(prophet_f1, 3),
    'lift_vs_single':    np.round(np.array(prophet_f1) - np.array(gmm_s_f1), 3),
    'lift_vs_cascade':   np.round(np.array(prophet_f1) - np.array(gmm_c_f1), 3),
    'GMM_single_BA':     np.round(gmm_s_ba, 3),
    'GMM_cascade_BA':    np.round(gmm_c_ba, 3),
    'Prophet_BA':        np.round(prophet_ba, 3),
})
comp.to_csv('prophet_vs_gmm_table.csv', index=False)
comp


## Summary table by noise family


In [ ]:
summary_cols = (
    [f'tol_{k}_f1_det'    for k in TOLERANCES] +
    [f'tol_{k}_f1_signed' for k in TOLERANCES] +
    [f'tol_{k}_dir_acc'   for k in TOLERANCES] +
    ['balanced_acc']
)
summary = (df_prophet.groupby('noise_family')[summary_cols].mean()
             .reindex(NOISE_FAMILIES).round(3))
summary.to_csv('prophet_summary_by_family.csv')
summary


## PELT baseline


In [ ]:
!pip install ruptures -q
import ruptures as rpt
import time

PELT_MODEL    = 'l2'
PELT_MIN_SIZE = 5
PELT_JUMP     = 1
SIGN_WINDOW   = 20


def estimate_sigma_from_diff(X, train_part=0.7):
    n = len(X); split = int(n * train_part)
    d = np.diff(X[:split])
    med = np.median(d); mad = np.median(np.abs(d - med))
    return float(max(1.4826 * mad / np.sqrt(2.0), 1e-9))


def pelt_predict(X, train_part=0.7, sign_window=SIGN_WINDOW,
                 model=PELT_MODEL, min_size=PELT_MIN_SIZE, jump=PELT_JUMP):
    n = len(X)
    sigma_hat = estimate_sigma_from_diff(X, train_part=train_part)
    pen = (sigma_hat ** 2) * np.log(n)
    algo = rpt.Pelt(model=model, min_size=min_size, jump=jump).fit(X.reshape(-1, 1))
    bkps = algo.predict(pen=pen)
    if len(bkps) and bkps[-1] == n:
        bkps = bkps[:-1]
    y_hat = np.zeros(n, dtype=int)
    h = sign_window
    for tau in bkps:
        lo = max(0, tau - h); hi = min(n, tau + h)
        if hi - tau < 1 or tau - lo < 1: continue
        s = int(np.sign(X[tau:hi].mean() - X[lo:tau].mean()))
        if s != 0:
            y_hat[tau] = s
    return {'y_hat': y_hat, 'bkps': np.asarray(bkps, dtype=int),
            'sigma_hat': sigma_hat, 'pen': pen}


_t0 = time.perf_counter()
_demo_df = make_series('red_a2', 0)
_X = _demo_df['x'].to_numpy(); _y = _demo_df['state'].to_numpy()
_pred = pelt_predict(_X, train_part=TRAIN_PART, sign_window=SIGN_WINDOW)
_m = ternary_metrics(_y, _pred['y_hat'], tolerances=TOLERANCES)
_dt = time.perf_counter() - _t0
print(f'PELT smoke test (red_a2, seed 0): '
      f'F1_signed(k=5) = {_m["tol_5_f1_signed"]:.3f}, '
      f'BalAcc = {_m["balanced_acc"]:.3f}, '
      f'n_pred = {int((_pred["y_hat"] != 0).sum())}, '
      f'wall {_dt:.2f}s')

### PELT under three parameter regimes


In [ ]:
_diag_settings = {
    'fast (j=5,m=20)':         dict(jump=5, min_size=20),
    'intermediate (j=1,m=20)': dict(jump=1, min_size=20),
    'honest (j=1,m=5)':        dict(jump=1, min_size=5),
}

_diag_rows = []
for fam in NOISE_FAMILIES:
    df_one = make_series(f'{fam}_a2', 0)
    X = df_one['x'].to_numpy()
    y = df_one['state'].to_numpy()
    sigma_hat = estimate_sigma_from_diff(X, train_part=TRAIN_PART)
    base_pen  = (sigma_hat ** 2) * np.log(len(X))
    for label, kw in _diag_settings.items():
        algo = rpt.Pelt(model=PELT_MODEL, **kw).fit(X.reshape(-1, 1))
        bkps = algo.predict(pen=base_pen)
        if len(bkps) and bkps[-1] == len(X):
            bkps = bkps[:-1]
        y_hat = np.zeros(len(X), dtype=int)
        for tau in bkps:
            lo = max(0, tau - SIGN_WINDOW); hi = min(len(X), tau + SIGN_WINDOW)
            if hi - tau < 1 or tau - lo < 1: continue
            s = int(np.sign(X[tau:hi].mean() - X[lo:tau].mean()))
            if s != 0:
                y_hat[tau] = s
        m = ternary_metrics(y, y_hat, tolerances=[2, 5])
        _diag_rows.append({
            'family':   fam,
            'settings': label,
            'F1_k2':    m['tol_2_f1_signed'],
            'F1_k5':    m['tol_5_f1_signed'],
            'BalAcc':   m['balanced_acc'],
            'n_pred':   int((y_hat != 0).sum()),
        })

df_diag = pd.DataFrame(_diag_rows)
diag_pivot = df_diag.pivot_table(
    index='family', columns='settings',
    values=['F1_k2', 'F1_k5', 'BalAcc']
).reindex(NOISE_FAMILIES).round(3)
print('PELT under three regimes (one representative series per family, amp=2):')
diag_pivot


## Full PELT benchmark loop


In [ ]:
rows_pelt = []
_first_exc_pelt = None
_t0 = time.perf_counter()

for noise_name, seed, df in tqdm(iter_dataset(),
                                   total=len(NOISE_CONFIGS)*len(RANDOM_SEEDS),
                                   desc='PELT ternary'):
    X = df['x'].to_numpy(); y = df['state'].to_numpy()
    try:
        pred = pelt_predict(X, train_part=TRAIN_PART, sign_window=SIGN_WINDOW)
        y_hat = pred['y_hat']
        m = ternary_metrics(y, y_hat, tolerances=TOLERANCES)
        m['sigma_hat']  = pred['sigma_hat']
        m['pen']        = pred['pen']
        m['n_pred_cps'] = int((y_hat != 0).sum())
        m['n_true_cps'] = int(((np.diff(y) != 0).sum()))
    except Exception as e:
        if _first_exc_pelt is None: _first_exc_pelt = e
        m = {}
    rows_pelt.append({'noise': noise_name, 'seed': seed,
                       'noise_family': noise_family(noise_name),
                       'amplitude': extract_amp(noise_name), **m})

if _first_exc_pelt:
    import traceback
    traceback.print_exception(type(_first_exc_pelt), _first_exc_pelt,
                                _first_exc_pelt.__traceback__)

df_pelt = pd.DataFrame(rows_pelt)
df_pelt.to_csv('pelt_ternary_results.csv', index=False)
_dt = time.perf_counter() - _t0
print(f'Done: {len(df_pelt)} rows, {df_pelt["tol_5_f1_det"].notna().sum()} valid, '
      f'total wall {_dt:.1f}s.')
df_pelt.head(3)

## PELT per-family summary


In [ ]:
summary_cols_pelt = (
    [f'tol_{k}_f1_det'    for k in TOLERANCES] +
    [f'tol_{k}_f1_signed' for k in TOLERANCES] +
    [f'tol_{k}_dir_acc'   for k in TOLERANCES] +
    ['balanced_acc']
)
summary_pelt = (df_pelt.groupby('noise_family')[summary_cols_pelt].mean()
                  .reindex(NOISE_FAMILIES).round(3))
summary_pelt.to_csv('pelt_summary_by_family.csv')
summary_pelt

## PELT vs GMM cascade vs Prophet


In [ ]:
tol = 5
fams = NOISE_FAMILIES

pelt_f1    = df_pelt.groupby('noise_family')[f'tol_{tol}_f1_signed'].mean().reindex(fams)
pelt_ba    = df_pelt.groupby('noise_family')['balanced_acc'].mean().reindex(fams)
prophet_f1 = df_prophet.groupby('noise_family')[f'tol_{tol}_f1_signed'].mean().reindex(fams)
prophet_ba = df_prophet.groupby('noise_family')['balanced_acc'].mean().reindex(fams)
gmm_s_f1   = [PAPER_TABLE2[f]['gmm_single_f1']  for f in fams]
gmm_s_ba   = [PAPER_TABLE2[f]['gmm_single_ba']  for f in fams]
gmm_c_f1   = [PAPER_TABLE2[f]['gmm_cascade_f1'] for f in fams]
gmm_c_ba   = [PAPER_TABLE2[f]['gmm_cascade_ba'] for f in fams]

comparison = pd.DataFrame({
    'PELT_F1sgn'         : pelt_f1.round(3).values,
    'PELT_BalAcc'        : pelt_ba.round(3).values,
    'GMM_single_F1sgn'   : np.round(gmm_s_f1, 3),
    'GMM_single_BalAcc'  : np.round(gmm_s_ba, 3),
    'GMM_cascade_F1sgn'  : np.round(gmm_c_f1, 3),
    'GMM_cascade_BalAcc' : np.round(gmm_c_ba, 3),
    'Prophet_F1sgn'      : prophet_f1.round(3).values,
    'Prophet_BalAcc'     : prophet_ba.round(3).values,
}, index=[FAMILY_BETA_PLAIN[f].replace('\n', ' ') for f in fams])
comparison.index.name = 'Noise family'
comparison.to_csv('all_methods_comparison.csv')
print('Saved -> all_methods_comparison.csv')
comparison

## Signed $F_1$ and balanced accuracy by noise family, all four methods


In [ ]:
x = np.arange(len(fams)); w = 0.20

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
ax = axes[0]
ax.bar(x - 1.5*w, gmm_s_f1,           w, color='lightgray', edgecolor='black', label='GMM single-stage')
ax.bar(x - 0.5*w, gmm_c_f1,           w, color='indianred', edgecolor='black', label='GMM cascade')
ax.bar(x + 0.5*w, pelt_f1.values,     w, color='gold',      edgecolor='black', label='PELT (+ sign)')
ax.bar(x + 1.5*w, prophet_f1.values,  w, color='steelblue', edgecolor='black', label='Prophet')
ax.set_xticks(x); ax.set_xticklabels([FAMILY_BETA_PLAIN[f] for f in fams], fontsize=9)
ax.set_ylabel(f'$F_{{1,\mathrm{{signed}}}}$  (tolerance $k={tol}$)', fontsize=11)
ax.set_ylim(0, 1.0); ax.grid(axis='y', alpha=0.3)
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.set_title(f'Signed $F_1$ by noise family (k = {tol})', fontsize=11)

ax = axes[1]
ax.bar(x - 1.5*w, gmm_s_ba,           w, color='lightgray', edgecolor='black')
ax.bar(x - 0.5*w, gmm_c_ba,           w, color='indianred', edgecolor='black')
ax.bar(x + 0.5*w, pelt_ba.values,     w, color='gold',      edgecolor='black')
ax.bar(x + 1.5*w, prophet_ba.values,  w, color='steelblue', edgecolor='black')
ax.axhline(1/3, ls=':', color='gray', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([FAMILY_BETA_PLAIN[f] for f in fams], fontsize=9)
ax.set_ylabel('Balanced accuracy (3-class macro recall)', fontsize=11)
ax.set_ylim(0, 1.0); ax.grid(axis='y', alpha=0.3)
ax.set_title('Pointwise balanced accuracy', fontsize=11)

plt.tight_layout()
plt.savefig('prophet_fig11_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()

## PELT sensitivity to the penalty multiplier


In [ ]:
alphas = [0.5, 1.0, 2.0, 4.0]
rep_seed = 0

rows_sens = []
for fam in NOISE_FAMILIES:
    noise_name = f'{fam}_a2'
    df_one = make_series(noise_name, rep_seed)
    X = df_one['x'].to_numpy(); y = df_one['state'].to_numpy()
    sigma_hat = estimate_sigma_from_diff(X, train_part=TRAIN_PART)
    base_pen = (sigma_hat ** 2) * np.log(len(X))
    for a in alphas:
        pen = a * base_pen
        algo = rpt.Pelt(model=PELT_MODEL, min_size=PELT_MIN_SIZE, jump=PELT_JUMP)
        algo = algo.fit(X.reshape(-1, 1))
        bkps = algo.predict(pen=pen)
        if len(bkps) and bkps[-1] == len(X):
            bkps = bkps[:-1]
        y_hat = np.zeros(len(X), dtype=int)
        for tau in bkps:
            lo = max(0, tau - SIGN_WINDOW); hi = min(len(X), tau + SIGN_WINDOW)
            if hi - tau < 1 or tau - lo < 1: continue
            s = int(np.sign(X[tau:hi].mean() - X[lo:tau].mean()))
            if s != 0: y_hat[tau] = s
        m = ternary_metrics(y, y_hat, tolerances=TOLERANCES)
        rows_sens.append({'family': fam, 'alpha': a, 'noise_name': noise_name,
                            'F1sgn': m['tol_5_f1_signed'],
                            'BalAcc': m['balanced_acc'],
                            'n_pred': int((y_hat != 0).sum())})

df_sens = pd.DataFrame(rows_sens)
pivot_f1 = df_sens.pivot(index='family', columns='alpha', values='F1sgn').reindex(NOISE_FAMILIES).round(3)
pivot_n  = df_sens.pivot(index='family', columns='alpha', values='n_pred').reindex(NOISE_FAMILIES)
print('Signed F1 at k=5 vs penalty multiplier alpha (one representative series per family):')
print(pivot_f1)
print()
print('Number of detected change points vs alpha:')
print(pivot_n)